# Workshop 3A: Symmetric Cryptography Lab (Solutions Guide)

**Topic**: Concurrency, Operating Systems, & Security — Symmetric Encryption  
**Target Duration**: 20–30 minutes  
**Objective**: Implement historic ciphers from scratch, learn bitwise XOR encryption, and write a modern, production-grade AES encryption script using Python libraries.

---  
## Exercise A: The Caesar Cipher (Letter Shifting)

The Caesar Cipher is one of the oldest known encryption techniques. It operates by replacing each letter in the plaintext with a letter a fixed number of positions down the alphabet.

### 1. Run the Encryption Function
Run the cell below to define the `caesar_encrypt` function and encrypt a sample message.

In [ ]:
def caesar_encrypt(plaintext: str, shift: int) -> str:
    ciphertext = ""
    for char in plaintext:
        if char.isalpha():
            # Shift characters and handle alphabetical wrapping
            start = ord('A') if char.isupper() else ord('a')
            shifted = (ord(char) - start + shift) % 26 + start
            ciphertext += chr(shifted)
        else:
            # Leave punctuation, numbers, and spaces unchanged
            ciphertext += char
    return ciphertext

# Test Encryption
msg = "Cryptography is fun!"
key = 3
encrypted_msg = caesar_encrypt(msg, key)
print(f"Original:  {msg}")
print(f"Encrypted: {encrypted_msg}") # Expected: "Fubswrjudskb lv ixq!"

### 2. Implement the Decryption Function
Write a function `caesar_decrypt(ciphertext: str, shift: int) -> str` that reverses the Caesar cipher transformation.  
*Hint: Decrypting with a key of `shift` is equivalent to encrypting with a key of `-shift`.*

In [ ]:
def caesar_decrypt(ciphertext: str, shift: int) -> str:
    # Decrypting is just shifting in the negative direction
    return caesar_encrypt(ciphertext, -shift)

# Verification Test
decrypted_msg = caesar_decrypt(encrypted_msg, key)
print(f"Decrypted: {decrypted_msg}")
assert decrypted_msg == msg, "Decryption failed!"

### 3. Brute-Force Attack
Imagine you intercepted the encrypted message `"Fubswrjudskb lv ixq!"` but do not know the key (shift size).  
Write a simple loop that tests all possible shifts from `1` to `25` to decrypt the ciphertext and output the results. Locate the correct shift.

In [ ]:
intercepted_ciphertext = "Fubswrjudskb lv ixq!"

# Test all possible shifts (1 to 25)
for shift_guess in range(1, 26):
    candidate = caesar_decrypt(intercepted_ciphertext, shift_guess)
    print(f"Shift {shift_guess:2d}: {candidate}")

---  
## Exercise B: The One-Time Pad (XOR Cipher & Perfect Secrecy)

A One-Time Pad (OTP) uses a key that is completely random, of equal length to the message, and used only once. Under these conditions, the cipher is mathematically unbreakable (perfect secrecy).

OTP utilizes the bitwise XOR (`^`) operator. Let's build a simple Python XOR cipher.

### 1. Implement OTP XOR Logic
Complete the `otp_xor` function below. You need to loop through the bytes of the message and key and perform a bitwise XOR (`^`) between each pair.

In [ ]:
import os

def generate_otp_key(length: int) -> bytes:
    # Generates cryptographically secure random bytes
    return os.urandom(length)

def otp_xor(message: bytes, key: bytes) -> bytes:
    if len(message) != len(key):
        raise ValueError("Key and message must be of equal length!")
    
    # XOR each byte of the message with the corresponding key byte
    return bytes(m ^ k for m, k in zip(message, key)) 

# Test Implementation
plaintext_bytes = b"CONFIDENTIAL"
otp_key = generate_otp_key(len(plaintext_bytes))

ciphertext_bytes = otp_xor(plaintext_bytes, otp_key)
decrypted_bytes = otp_xor(ciphertext_bytes, otp_key)

print(f"Plaintext:  {plaintext_bytes}")
print(f"Ciphertext: {ciphertext_bytes.hex()}")
print(f"Decrypted:  {decrypted_bytes.decode()}")

assert decrypted_bytes == plaintext_bytes, "OTP Decryption failed!"

### Discussion Questions & Solutions:
1. **Why is the One-Time Pad mathematically unbreakable?**
   * **Answer**: Because the key is completely random and of equal length to the message, the ciphertext contains zero information about the plaintext. For any ciphertext of length $L$, *every possible plaintext* of length $L$ is equally likely given the correct key (e.g. any ciphertext could decrypt to any word of that length with the right key choice, meaning an attacker with infinite computation gains no information).
2. **What is the major practical challenge of using it in distributed networks?**
   * **Answer**: The key distribution problem. The parties must securely exchange a key that is as long as the message before transmission. If a secure channel existed to send that key, they could have just sent the message over that channel instead.

---  
## Exercise C: Practical AES-GCM (Real-world Encryption)

In production environments, developers must **never** roll their own ciphers from scratch due to implementation vulnerabilities (e.g. side-channel attacks, padding oracle attacks). Instead, we use standardized libraries.

We will use the standard Python `cryptography` library to encrypt and decrypt a message using **AES-GCM** (Advanced Encryption Standard in Galois/Counter Mode). AES-GCM provides both confidentiality (encryption) and integrity (tamper verification).

### 1. Install the Library
First, make sure the library is installed.

In [ ]:
!pip install cryptography

### 2. Complete the AES-GCM Decryption
Review the encryption phase below, then write the decryption phase using the `.decrypt(...)` method.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# 1. Generate a random 256-bit AES key
key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(key)

# 2. Generate a random 96-bit Initialization Vector (IV) / Nonce
# This ensures that encrypting the same message twice results in different ciphertexts
nonce = os.urandom(12) 

# 3. Plaintext message
plaintext = b"Symmetric AES-GCM is secure!"

# 4. Encrypt
ciphertext = aesgcm.encrypt(nonce, plaintext, associated_data=None)
print(f"Ciphertext (hex): {ciphertext.hex()}")

# 5. Decrypt
decrypted = aesgcm.decrypt(nonce, ciphertext, associated_data=None)

print(f"Decrypted: {decrypted.decode()}")
assert decrypted == plaintext, "AES Decryption failed!"

### Discussion Questions & Solutions:
1. **What is the role of the `nonce` (number used once) / Initialization Vector in AES-GCM?**
   * **Answer**: It ensures that identical plaintexts encrypted with the same key produce completely different ciphertexts. This prevents attackers from detecting patterns or identifying duplicate messages.
2. **What happens to the security of the cipher if a developer reuses the same `nonce` with the same key for multiple messages?**
   * **Answer**: It is catastrophic. In GCM mode, reusing a nonce allows an attacker to recover the key stream, which completely breaks confidentiality (allowing them to decrypt all messages). It also breaks the authentication tag validation, allowing attackers to modify/forge ciphertexts.